# Sprint 1 — Data Pipeline End-to-End Check

Verifies the leakage-safe pipeline: Football-Data raw → canonical → `(X, y, meta)` feature matrix.
Run the cells top-to-bottom. The key one is **Cell 2** (NaN check) — X must be fully dense before it goes into a `RandomForestClassifier`.

### Cell 1 — build the matrix

In [1]:
import sys, json
from pathlib import Path
import pandas as pd

# Make sure the project root is importable (this notebook lives in notebooks/)
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.features.build_features import build_features

# Don't overwrite the committed schema artifact from a notebook run
X, y, meta = build_features(save_schema=False)
print(f"X: {X.shape}   y: {y.shape}   meta: {meta.shape}")
X.head()

X: (1041, 34)   y: (1041,)   meta: (1041, 5)


,home_rolling_win_rate,home_rolling_goals_for_avg,home_rolling_goals_against_avg,home_rolling_win_rate_home,home_rolling_win_rate_away,home_rolling_goals_for_home,home_rolling_goals_for_away,home_rolling_goals_against_home,home_rolling_goals_against_away,home_rolling_points_per_game,...,away_rolling_goal_difference_per_game,away_rolling_clean_sheet_rate,away_days_rest,away_is_new_team,home_league_position,away_league_position,home_position_diff,elo_home_pre,elo_away_pre,elo_diff
0,0.382086,1.512818,1.506093,0.442432,0.327924,1.640207,1.394061,1.38206,1.62746,1.385383,...,0.006725,0.230263,7.0,1,10,10,0,1450.0,1450.0,0.0
1,0.382086,1.512818,1.506093,0.442432,0.327924,1.640207,1.394061,1.38206,1.62746,1.385383,...,0.006725,0.230263,7.0,1,10,10,0,1450.0,1450.0,0.0
2,0.382086,1.512818,1.506093,0.442432,0.327924,1.640207,1.394061,1.38206,1.62746,1.385383,...,0.006725,0.230263,7.0,1,10,10,0,1450.0,1450.0,0.0
3,0.382086,1.512818,1.506093,0.442432,0.327924,1.640207,1.394061,1.38206,1.62746,1.385383,...,0.006725,0.230263,7.0,1,10,10,0,1450.0,1450.0,0.0
4,0.382086,1.512818,1.506093,0.442432,0.327924,1.640207,1.394061,1.38206,1.62746,1.385383,...,0.006725,0.230263,7.0,1,10,10,0,1450.0,1450.0,0.0


### Cell 2 — shapes, dtypes, and the all-important NaN check

In [2]:
print("Feature columns:", list(X.columns))
print("\nDtypes:\n", X.dtypes)

nan_counts = X.isna().sum()
print("\nColumns with NaNs (must be empty for sklearn):")
print(nan_counts[nan_counts > 0] if nan_counts.any() else "✅ None — X is fully dense")

Feature columns: ['home_rolling_win_rate', 'home_rolling_goals_for_avg', 'home_rolling_goals_against_avg', 'home_rolling_win_rate_home', 'home_rolling_win_rate_away', 'home_rolling_goals_for_home', 'home_rolling_goals_for_away', 'home_rolling_goals_against_home', 'home_rolling_goals_against_away', 'home_rolling_points_per_game', 'home_rolling_goal_difference_per_game', 'home_rolling_clean_sheet_rate', 'home_days_rest', 'home_is_new_team', 'away_rolling_win_rate', 'away_rolling_goals_for_avg', 'away_rolling_goals_against_avg', 'away_rolling_win_rate_home', 'away_rolling_win_rate_away', 'away_rolling_goals_for_home', 'away_rolling_goals_for_away', 'away_rolling_goals_against_home', 'away_rolling_goals_against_away', 'away_rolling_points_per_game', 'away_rolling_goal_difference_per_game', 'away_rolling_clean_sheet_rate', 'away_days_rest', 'away_is_new_team', 'home_league_position', 'away_league_position', 'home_position_diff', 'elo_home_pre', 'elo_away_pre', 'elo_diff']

Dtypes:
 home_rol

### Cell 3 — label distribution (sanity-check class balance)

In [3]:
print("y label counts:\n", y.value_counts())
print("\ny proportions:\n", (y.value_counts(normalize=True).round(3)))
# Expect home-win the largest class (home advantage), draws smallest-ish.

y label counts:
 result
HW    450
AW    342
D     249
Name: count, dtype: int64[pyarrow]

y proportions:
 result
HW    0.432
AW    0.329
D     0.239
Name: proportion, dtype: double[pyarrow]


### Cell 4 — alignment + temporal ordering (the leakage-relevant check)

In [4]:
assert len(X) == len(y) == len(meta), "X/y/meta length mismatch"
assert list(X.index) == list(meta.index) == list(y.index), "indices not aligned"

meta = meta.copy()
meta["date"] = pd.to_datetime(meta["date"])
print("Date range:", meta["date"].min().date(), "→", meta["date"].max().date())
print("Seasons:", sorted(meta['season'].unique()))
print("Rows monotonic in date:", meta["date"].is_monotonic_increasing)
# If not monotonic, a time-ordered sort is needed before a walk-forward split (Sprint 2).

Date range: 2023-08-11 → 2026-03-01
Seasons: [np.int64(2023), np.int64(2024), np.int64(2025)]
Rows monotonic in date: True


### Cell 5 — leakage spot-check on the earliest match

In [5]:
# The very first match in the data can have NO prior history.
# Its rolling features should be imputed/neutral, never derived from later games.
first = meta.sort_values("date").iloc[0]
row = X.loc[meta["match_id"] == first["match_id"]].iloc[0]
print(f"Earliest match: {first['home_team']} vs {first['away_team']} ({first['date'].date()})")
print(row[[c for c in X.columns if "rolling" in c or "elo" in c or "position" in c]])
# Elo should sit near the 1450 start; rolling rates near their imputed neutral values.

Earliest match: Burnley FC vs Manchester City FC (2023-08-11)
home_rolling_win_rate                       0.382086
home_rolling_goals_for_avg                  1.512818
home_rolling_goals_against_avg              1.506093
home_rolling_win_rate_home                  0.442432
home_rolling_win_rate_away                  0.327924
home_rolling_goals_for_home                 1.640207
home_rolling_goals_for_away                 1.394061
home_rolling_goals_against_home             1.382060
home_rolling_goals_against_away             1.627460
home_rolling_points_per_game                1.385383
home_rolling_goal_difference_per_game       0.006725
home_rolling_clean_sheet_rate               0.230263
away_rolling_win_rate                       0.382086
away_rolling_goals_for_avg                  1.512818
away_rolling_goals_against_avg              1.506093
away_rolling_win_rate_home                  0.442432
away_rolling_win_rate_away                  0.327924
away_rolling_goals_for_home          

### Cell 6 — a real temporal split (how Sprint 2 will train) + the validation artifacts

In [6]:
# Train on all-but-last season, "predict" the last — proves the matrix supports a clean time split.
last_season = sorted(meta["season"].unique())[-1]
train_idx = meta.index[meta["season"] < last_season]
test_idx  = meta.index[meta["season"] == last_season]
print(f"Train: {len(train_idx)} matches (< {last_season})   Test: {len(test_idx)} matches ({last_season})")
print("Max train date < min test date:",
      meta.loc[train_idx, "date"].max() < meta.loc[test_idx, "date"].min())

# Show the data-quality artifacts the ingest step produced
report = json.loads((ROOT / "data/processed/validation_report.json").read_text())
print("\nValidation report:", json.dumps({k: report[k] for k in
      ["total_rows","rows_kept","rows_dropped","dropped_by_reason"]}, indent=2))
schema = json.loads((ROOT / "artifacts/feature_schema.json").read_text())
print("Locked schema n_features:", schema["n_features"])

Train: 760 matches (< 2025)   Test: 281 matches (2025)
Max train date < min test date: True

Validation report: {
  "total_rows": 1140,
  "rows_kept": 1041,
  "rows_dropped": 99,
  "dropped_by_reason": {
    "missing_goals": 99
  }
}
Locked schema n_features: 34
